# 04 — E06 Neutering Iteration Smoke

One manual neutralization iteration for `calm_2021`:

`raw_summary → J1 Q1 identifiers → J1 Q3 baseline → N candidate → J1 Q3 candidate → preservation score → artifacts`.

This notebook is a research smoke artifact only. It does not write to the production database and does not run J2 or Q2 personas.

In [2]:
import json
import os
import sqlite3
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
load_dotenv(PROJECT_ROOT / ".env")

DB_PATH = "data/db/blind_prophet.db"
RUN_DATE = "2021-10-20"
POINT_LABEL = "calm_2021"
OUT_DIR = Path("data/runs/e06_smoke/calm_2021")

MODEL_N = "deepseek/deepseek-v4-pro"
MODEL_J1 = "qwen/qwen3.5-397b-a17b"
# If OpenRouter availability changes, adjust model IDs here only. No provider switching is implemented in this smoke notebook.

TEMPERATURE_J1 = 0.1
TEMPERATURE_N = 0.3

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

## Load raw meta-summary

In [3]:
def load_summary(db_path: str, run_date: str) -> str:
    path = Path(db_path)
    if not path.exists():
        raise FileNotFoundError(f"SQLite DB not found: {path}")

    with sqlite3.connect(path) as conn:
        row = conn.execute(
            "SELECT summary FROM summaries WHERE run_date = ?",
            (run_date,),
        ).fetchone()

    if row is None or not row[0]:
        raise LookupError(f"No summary found in table 'summaries' for run_date={run_date!r}")
    return row[0]


raw_summary = load_summary(DB_PATH, RUN_DATE)
print(f"Loaded summary for {POINT_LABEL} ({RUN_DATE})")
print("Raw length:", len(raw_summary))

Loaded summary for calm_2021 (2021-10-20)
Raw length: 8860


## LLM and JSON helpers

In [4]:
def call_llm(model: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY is not set in the environment")

    client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    content = response.choices[0].message.content
    if content is None:
        raise RuntimeError(f"Model {model} returned empty content")
    return content.strip()


def extract_json(raw_text: str) -> dict:
    text = raw_text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError as exc:
        preview = text[:500]
        raise ValueError(f"Could not parse strict JSON after optional fence stripping: {exc}\nPreview:\n{preview}") from exc

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected a JSON object, got {type(parsed).__name__}")
    return parsed

## Prompt contracts

In [5]:
Q1_EVIDENCE_SYSTEM = """
You are J1, an in-loop leakage evidence judge for a Russian macroeconomic summary.

Task: identify phrases that could let a model infer the true historical period of the summary.
Return strict JSON only. Do not include markdown, commentary, or extra keys.

Output schema:
{
  "identifiers": [
    {
      "id": "id_001",
      "class": "number | event | time_anchor",
      "pattern_description": "...",
      "anchor_phrase": "...",
      "suggested_strategy": "..."
    }
  ]
}

Identifier classes:
- number: distinctive levels, coordinates, rates, prices, or quantities that act as period fingerprints.
- event: unique or near-unique factual configurations, named events, policy episodes, sanctions, conflicts, institutions, places, or market episodes.
- time_anchor: explicit or implicit time coordinates, chronology labels, dates, months, years, holidays, or survey-window clues.

Strict leakage rule: you must NOT leak the true period in your answer.
- No years.
- No month names.
- No exact dates.
- No unique historical event names.
- No euphemisms or aliases that identify the exact episode.

Describe only leakage classes and mechanisms. The anchor_phrase may quote or minimally normalize the source phrase, but if the source phrase contains a forbidden exact period marker, mask it with a generic placeholder.
Use stable ids id_001, id_002, ... .
""".strip()


Q3_SIGNALS_SYSTEM = """
You are J1, an in-loop forecast-signal judge for a Russian macroeconomic summary.

Task: extract only forecast-relevant inflation expectation signals that are recoverable from the summary.
Return strict JSON only. Do not include markdown, commentary, or extra keys.

Output schema:
{
  "signals": [
    {
      "id": "sig_001",
      "category": "monetary_policy | supply_shock_food | supply_shock_energy | currency_pressure | fiscal_expansion | demand_shift | regulated_prices | inflation_print",
      "direction": "up | down | mixed | neutral",
      "strength": 1,
      "anchor_phrase": "..."
    }
  ],
  "summary_direction": "up | down | mixed | neutral",
  "summary_confidence": 0.0
}

Use the closed category list only:
- monetary_policy
- supply_shock_food
- supply_shock_energy
- currency_pressure
- fiscal_expansion
- demand_shift
- regulated_prices
- inflation_print

No extra categories are allowed.
direction must be exactly one of: up, down, mixed, neutral.
strength must be an integer from 1 to 3, where 1=weak, 2=medium, 3=strong.
anchor_phrase must be copied or minimally normalized from the summary.
summary_confidence must be a number from 0.0 to 1.0.
Use stable ids sig_001, sig_002, ... .
""".strip()


N_REWRITER_SYSTEM = """
You are N, a neutralizing rewriter for a Russian macroeconomic summary.

Task: rewrite the previous summary to reduce period-identifying markers while preserving forecast-relevant inflation expectation signals.
Return plain text only. Do not return JSON. Do not include markdown fences or explanations.

Invariants:
- Length must stay within +/-20% of the input summary.
- Preserve event order.
- Mask identifier anchors according to the suggested strategies from J1 Q1-evidence.
- Preserve Q3 signals so that category, direction, and strength +/-1 remain recoverable.
- Levels and numeric coordinates should be generalized.
- Deltas and changes may be preserved when they carry forecast signal.
- Absolute dates should be converted to relative references or removed.
- Events should be rewritten through mechanism and role, without names, places, or dates when those identify the period.
- Do not add new facts.
- Do not turn the text into a generic macro summary; keep the causal structure and sector detail.
""".strip()

## Scoring

In [6]:
def q3_preservation_score(baseline: dict, candidate: dict) -> float:
    baseline_signals = baseline.get("signals", [])
    candidate_signals = candidate.get("signals", [])
    if not baseline_signals:
        return 1.0

    candidate_by_category = {}
    for signal in candidate_signals:
        candidate_by_category.setdefault(signal.get("category"), []).append(signal)

    matched = []
    direction_matches = 0
    strength_consistent = 0

    for base_signal in baseline_signals:
        category = base_signal.get("category")
        candidates = candidate_by_category.get(category, [])
        if not candidates:
            continue
        cand_signal = candidates.pop(0)
        matched.append((base_signal, cand_signal))

        if cand_signal.get("direction") == base_signal.get("direction"):
            direction_matches += 1

        try:
            base_strength = int(base_signal.get("strength"))
            cand_strength = int(cand_signal.get("strength"))
        except (TypeError, ValueError):
            continue
        if abs(base_strength - cand_strength) <= 1:
            strength_consistent += 1

    recall = len(matched) / len(baseline_signals)
    if not matched:
        return 0.0

    direction_match_rate = direction_matches / len(matched)
    strength_consistency_rate = strength_consistent / len(matched)
    return recall * direction_match_rate * strength_consistency_rate

## One manual iteration

In [7]:
q1_raw_text = call_llm(
    model=MODEL_J1,
    system_prompt=Q1_EVIDENCE_SYSTEM,
    user_prompt=f"Summary:\n\n{raw_summary}",
    temperature=TEMPERATURE_J1,
)
q1_identifiers_raw = extract_json(q1_raw_text)

q3_baseline_text = call_llm(
    model=MODEL_J1,
    system_prompt=Q3_SIGNALS_SYSTEM,
    user_prompt=f"Summary:\n\n{raw_summary}",
    temperature=TEMPERATURE_J1,
)
q3_baseline_raw = extract_json(q3_baseline_text)

rewriter_user_prompt = "\n\n".join(
    [
        "Previous summary:",
        raw_summary,
        "J1 Q1-evidence identifiers to mask:",
        json.dumps(q1_identifiers_raw.get("identifiers", []), ensure_ascii=False, indent=2),
        "J1 Q3 signals_to_preserve:",
        json.dumps(q3_baseline_raw.get("signals", []), ensure_ascii=False, indent=2),
    ]
)
candidate_summary = call_llm(
    model=MODEL_N,
    system_prompt=N_REWRITER_SYSTEM,
    user_prompt=rewriter_user_prompt,
    temperature=TEMPERATURE_N,
)

q3_candidate_text = call_llm(
    model=MODEL_J1,
    system_prompt=Q3_SIGNALS_SYSTEM,
    user_prompt=f"Summary:\n\n{candidate_summary}",
    temperature=TEMPERATURE_J1,
)
q3_candidate_iter1 = extract_json(q3_candidate_text)

q3_preservation = q3_preservation_score(q3_baseline_raw, q3_candidate_iter1)

## Diagnostics for manual review

In [8]:
raw_length = len(raw_summary)
candidate_length = len(candidate_summary)
length_ratio = candidate_length / raw_length if raw_length else 0.0

print("Raw length:", raw_length)
print("Candidate length:", candidate_length)
print("Length ratio:", round(length_ratio, 3))
print("Q3 preservation score:", round(q3_preservation, 3))

print("\n--- RAW SUMMARY PREVIEW ---\n")
print(raw_summary[:1000])

print("\n--- CANDIDATE SUMMARY PREVIEW ---\n")
print(candidate_summary[:1000])

print("\n--- Q1 IDENTIFIERS ---\n")
print(json.dumps(q1_identifiers_raw.get("identifiers", []), ensure_ascii=False, indent=2))

print("\n--- Q3 BASELINE SIGNALS ---\n")
print(json.dumps(q3_baseline_raw.get("signals", []), ensure_ascii=False, indent=2))

print("\n--- Q3 CANDIDATE SIGNALS ---\n")
print(json.dumps(q3_candidate_iter1.get("signals", []), ensure_ascii=False, indent=2))

Raw length: 8860
Candidate length: 8735
Length ratio: 0.986
Q3 preservation score: 0.875

--- RAW SUMMARY PREVIEW ---

Вот единый нарративный таймлайн, собранный из предоставленных тематических осей и сгруппированный по смысловым событийным узлам.

---

### Узел 1: Глобальный сырьевой шок и укрепление рубля (День 1 — День 3)

Период открывается серией исторических рекордов на сырьевых рынках. Цена газа в Европе достигает абсолютного максимума — $1920 за 1 тыс. куб. м, после чего, на фоне заявлений о готовности стабилизировать рынок, резко падает до $1110–1250. Цена энергетического угля в Европе превышает $300 за тонну, побив рекорд 2008 года. Нефть Brent поднимается выше $84 за баррель (впервые с октября 2018 года), WTI — до $80,52. Рублёвая цена российской нефти Urals обновляет исторический максимум, достигая 5,87 тыс. руб. за баррель (рост более чем на 23% с конца предыдущего периода).

На этом фоне происходит резкое укрепление рубля. Курс евро падает до 83 руб. (минимум с середины п

## Save deterministic artifacts

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "raw_summary": OUT_DIR / "raw_summary.md",
    "candidate_summary": OUT_DIR / "candidate_iter1.md",
    "q1_identifiers": OUT_DIR / "q1_identifiers_raw.json",
    "q3_baseline": OUT_DIR / "q3_baseline_raw.json",
    "q3_candidate": OUT_DIR / "q3_candidate_iter1.json",
    "iteration_record": OUT_DIR / "iteration_1_record.json",
}

paths["raw_summary"].write_text(raw_summary, encoding="utf-8")
paths["candidate_summary"].write_text(candidate_summary, encoding="utf-8")
paths["q1_identifiers"].write_text(json.dumps(q1_identifiers_raw, ensure_ascii=False, indent=2), encoding="utf-8")
paths["q3_baseline"].write_text(json.dumps(q3_baseline_raw, ensure_ascii=False, indent=2), encoding="utf-8")
paths["q3_candidate"].write_text(json.dumps(q3_candidate_iter1, ensure_ascii=False, indent=2), encoding="utf-8")

iteration_record = {
    "run_date": RUN_DATE,
    "point_label": POINT_LABEL,
    "iteration": 1,
    "input_condition": "raw",
    "model_roles": {
        "N": MODEL_N,
        "J1": MODEL_J1,
    },
    "raw_length": raw_length,
    "candidate_length": candidate_length,
    "length_ratio": length_ratio,
    "q3_preservation": q3_preservation,
    "paths": {
        "raw_summary": str(paths["raw_summary"]),
        "candidate_summary": str(paths["candidate_summary"]),
        "q1_identifiers": str(paths["q1_identifiers"]),
        "q3_baseline": str(paths["q3_baseline"]),
        "q3_candidate": str(paths["q3_candidate"]),
    },
    "manual_review_required": True,
}
paths["iteration_record"].write_text(json.dumps(iteration_record, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved artifacts:")
for key, path in paths.items():
    print(f"- {key}: {path}")

Saved artifacts:
- raw_summary: data/runs/e06_smoke/calm_2021/raw_summary.md
- candidate_summary: data/runs/e06_smoke/calm_2021/candidate_iter1.md
- q1_identifiers: data/runs/e06_smoke/calm_2021/q1_identifiers_raw.json
- q3_baseline: data/runs/e06_smoke/calm_2021/q3_baseline_raw.json
- q3_candidate: data/runs/e06_smoke/calm_2021/q3_candidate_iter1.json
- iteration_record: data/runs/e06_smoke/calm_2021/iteration_1_record.json
